# Cleaning the Data - Different Preprocessing Strategies

In [9]:
# imports

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# data

train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

In [11]:
# Save IDs and target before anything

print(f"Train shape: {train.shape}\nTest shape: {test.shape}")

train_ids  = train['Id']
test_ids   = test['Id']
y          = np.log1p(train['SalePrice'])

train = train.drop(columns=['Id', 'SalePrice'])
test  = test.drop(columns=['Id'])

n_train = len(train)
all_data = pd.concat([train, test], axis=0).reset_index(drop=True)

print(f"Combined shape: {all_data.shape}")

Train shape: (1460, 81)
Test shape: (1459, 80)
Combined shape: (2919, 79)


In [12]:
# These NaNs mean the feature is ABSENT — not missing data
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]

for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

print("✅ Domain-informed imputation done")

✅ Domain-informed imputation done


In [13]:
# LotFrontage: use median of the same Neighborhood
# Much smarter than global median!
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'] \
                                   .transform(lambda x: x.fillna(x.median()))

# Fallback if still missing (rare edge case)
all_data['LotFrontage'] = all_data['LotFrontage'].fillna(all_data['LotFrontage'].median())

print(f"✅ LotFrontage missing after group impute: {all_data['LotFrontage'].isnull().sum()}")

✅ LotFrontage missing after group impute: 0


In [14]:
mode_cols = ['Electrical', 'MSZoning', 'Utilities', 'Functional',
             'SaleType', 'KitchenQual', 'Exterior1st', 'Exterior2nd']

for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print("✅ Mode imputation done")

✅ Mode imputation done


In [15]:
remaining = all_data.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)

if len(remaining) == 0:
    print("✅ No missing values remaining!")
else:
    print("⚠️ Still missing:")
    print(remaining)

✅ No missing values remaining!


In [16]:
# These have a meaningful order: None < Po < Fa < TA < Gd < Ex
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ordinal_quality_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

for col in ordinal_quality_cols:
    all_data[col] = all_data[col].map(quality_map)

# Other ordinal mappings
all_data['BsmtExposure'] = all_data['BsmtExposure'].map(
    {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
)
all_data['BsmtFinType1'] = all_data['BsmtFinType1'].map(
    {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
)
all_data['BsmtFinType2'] = all_data['BsmtFinType2'].map(
    {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
)
all_data['GarageFinish'] = all_data['GarageFinish'].map(
    {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
)
all_data['LotShape'] = all_data['LotShape'].map(
    {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3}
)
all_data['LandSlope'] = all_data['LandSlope'].map(
    {'Sev': 0, 'Mod': 1, 'Gtl': 2}
)
all_data['PavedDrive'] = all_data['PavedDrive'].map(
    {'N': 0, 'P': 1, 'Y': 2}
)
all_data['Functional'] = all_data['Functional'].map(
    {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3,
     'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}
)

print("✅ Ordinal encoding done")

✅ Ordinal encoding done


In [17]:
# Remaining object columns have no meaningful order
remaining_cats = all_data.select_dtypes(include='object').columns.tolist()
print(f"Remaining categoricals to label encode: {remaining_cats}")

le = LabelEncoder()
for col in remaining_cats:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

print("✅ Label encoding done")

Remaining categoricals to label encode: ['MSZoning', 'Street', 'Alley', 'LandContour', 'Utilities', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
✅ Label encoding done


In [ ]:
X_train = all_data.iloc[:n_train].reset_index(drop=True)
X_test  = all_data.iloc[n_train:].reset_index(drop=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',  index=False)
y.to_csv('../data/processed/Y_train.csv',       index=False)

print(f"✅ Saved!")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   Y_train : {y.shape}")

✅ Saved!
   X_train : (1460, 79)
   X_test  : (1459, 79)
   y_train : (1460,)
